# Módulo 01 — Computação Numérica e Álgebra Linear com `NumPy`

## 1. O que é o `NumPy` e por que não usar apenas listas do Python?

No módulo anterior, vimos que as **listas** do Python são flexíveis: podem guardar textos e números misturados. Porém, em simulações de engenharia e Redes Neurais com milhões de cálculos, essa flexibilidade cobra um preço alto: **lentidão**.

O **`NumPy`** (*Numerical Python*) introduz a estrutura **`ndarray`** (*N-dimensional array*, ou arranjo multidimensional).
- **Analogia de Engenharia:** Se uma lista comum do Python é uma sacola flexível com objetos variados, o `ndarray` do `NumPy` é uma **viga de aço laminada**: rígida (todos os elementos têm o mesmo tipo numérico), compacta na memória e até **100 vezes mais rápida**!
- **Por que começar pelo `NumPy`?** Porque você já estudou **Vetores, Matrizes e Sistemas Lineares** na graduação — e toda Rede Neural Artificial nada mais é do que multiplicações e somas de matrizes do `NumPy`.

Vamos importar a biblioteca usando o apelido universal `np`:

In [ ]:
# Importa a biblioteca NumPy com o apelido padrão 'np'
import numpy as np

print("Versão do NumPy carregada:", np.__version__)

## 2. Criando Vetores (1D) e Matrizes (2D)

- **Vetor (Array 1D):** Uma série temporal de leituras de um sensor de temperatura ou as entradas $[x_1, x_2, x_3]$ de um neurônio.
- **Matriz (Array 2D):** Uma tabela numérica com **linhas** e **colunas** (ex.: matriz de rigidez de uma treliça ou matriz de pesos $\mathbf{W}$ de uma rede neural). Usamos `arr.shape` (*formato/dimensão*) para consultar `(n_linhas, n_colunas)`.

In [ ]:
# Criando um vetor 1D a partir de uma lista Python de leituras de pressão (bar)
vetor_pressao = np.array([2.5, 2.7, 2.6, 3.0])
print("Vetor de Pressão:", vetor_pressao)
print("Formato (shape) do vetor:", vetor_pressao.shape)  # (4,) = 4 elementos em 1 dimensão

# Criando uma matriz 2x3 (2 linhas e 3 colunas) de tensões mecânicas (MPa)
# Repare nos colchetes externos envolvendo as listas de cada linha: [[linha_0], [linha_1]]
matriz_esforcos = np.array([
    [10.0, 20.0, 30.0],
    [40.0, 50.0, 60.0]
])
print("\nMatriz de Esforços:\n", matriz_esforcos)
print("Formato (linhas, colunas):", matriz_esforcos.shape)

**Análise:** Enquanto o vetor tem formato `(4,)`, a matriz tem formato `(2, 3)` (sempre **linha primeiro, coluna depois**, exatamente como $M_{2 \times 3}$ na Álgebra Linear!).

### Geradores Automáticos de Sequências e Matrizes Especiais
Na engenharia, raramente digitamos matrizes na mão. O `NumPy` oferece geradores prontos:
1. `np.arange(inicio, fim, passo)`: Gera sequência numérica que **inclui `inicio` e para imediatamente antes de `fim`**.
2. `np.linspace(inicio, fim, quantidade)`: Divide um intervalo contínuo (ex.: `0` a `10` segundos) em `quantidade` pontos igualmente espaçados (**inclui as duas extremidades** — perfeito para eixos de tempo!).
3. `np.zeros((linhas, colunas))` e `np.eye(n)`: Criam matrizes nulas e a Matriz Identidade $\mathbf{I}_n$.

In [ ]:
# Sequência inteira de 5 até 11 (o limite superior 12 é exclusivo!)
sequencia = np.arange(5, 12)
print("Sequência com arange(5, 12):", sequencia)

# Eixo de tempo contínuo de 0.0 s até 1.0 s com exatos 5 instantes igualmente espaçados
tempo = np.linspace(0.0, 1.0, 5)
print("Instantes de tempo (s):", tempo)

# Matriz de zeros 2x4 (muito usada para inicializar acumuladores e gradientes)
zeros_2x4 = np.zeros((2, 4))
print("\nMatriz de Zeros 2x4:\n", zeros_2x4)

# Matriz Identidade 3x3 (1 na diagonal principal e 0 fora dela)
identidade = np.eye(3)
print("\nMatriz Identidade 3x3:\n", identidade)

### Números Aleatórios e Reprodutibilidade (`seed`)
Quando criamos uma Rede Neural, suas "válvulas" (**pesos sinápticos**) começam com valores **aleatórios pequenos** antes de aprenderem com os dados.
- Para que um experimento científico seja **reprodutível** (isto é, dê exatamente o mesmo resultado no seu computador e no do professor), fixamos uma **semente aleatória (`seed`)** no gerador moderno `np.random.default_rng(42)`.

In [ ]:
# Criamos um gerador aleatório com semente fixa (42) para resultados 100% reprodutíveis
rng = np.random.default_rng(seed=42)

# Sorteia 5 números inteiros entre 50 (inclusivo) e 100 (exclusivo)
inteiros_aleatorios = rng.integers(low=50, high=100, size=5)
print("Inteiros sorteados:", inteiros_aleatorios)

# Sorteia pesos reais uniformemente distribuídos entre -0.5 e +0.5 (padrão clássico em RNAs!)
pesos_iniciais = rng.uniform(low=-0.5, high=0.5, size=(2, 3))
print("\nMatriz de pesos iniciais 2x3 em [-0.5, 0.5]:\n", np.round(pesos_iniciais, 4))

**Análise:** Sempre que precisar sortear pesos iniciais de uma Rede Neural, lembre-se de `rng.uniform(-0.5, 0.5, ...)` — pesos pequenos evitam que o neurônio comece "saturado" (travado nos extremos).

---

## 3. Operações Vetorizadas (*Broadcasting*) e Estatísticas

O recurso de **Broadcasting** (*transmissão automática*) permite aplicar fórmulas matemáticas a um vetor inteiro **sem escrever laços `for`**. Lembra da conversão Celsius → Kelvin que exigiu 4 linhas de `for` no Módulo 00? Veja como fica em 1 linha no `NumPy`:

In [ ]:
# Vetor de temperaturas em graus Celsius (°C)
celsius = np.array([0.0, 25.0, 37.0, 100.0])

# O NumPy transmite ('broadcasts') a soma +273.15 para TODOS os elementos instantaneamente!
kelvin = celsius + 273.15
print("Temperaturas em Kelvin:", kelvin)

# Operações estatísticas diretas sobre o vetor
print(f"Média: {np.mean(celsius):.2f} °C | Desvio Padrão: {np.std(celsius):.2f} °C")
print(f"Mínimo: {np.min(celsius):.1f} °C | Máximo: {np.max(celsius):.1f} °C")

---

## 4. Álgebra Linear: Multiplicação de Matrizes (`@`) vs. Elemento a Elemento (`*`)

> **Atenção — Diferença Mais Importante para Redes Neurais:**
> - `A * B`: Multiplica posição por posição (*produto de Hadamard*).
> - `A @ B` (ou `np.dot(A, B)`): Faz a **multiplicação matricial real da Álgebra Linear** (linha pela coluna: $\sum a_{ik} b_{kj}$), usada para calcular o potencial $u = \mathbf{w} \cdot \mathbf{x}$ de cada neurônio!
> - `np.linalg.solve(A, b)`: Resolve sistemas lineares $\mathbf{A}\mathbf{x} = \mathbf{b}$ (como malhas elétricas de Kirchhoff ou equilíbrio de forças em treliças) com alta estabilidade numérica.

In [ ]:
# Exemplo: Sistema de equações de duas malhas elétricas (3*i1 + 1*i2 = 9  e  1*i1 + 2*i2 = 8)
A = np.array([
    [3.0, 1.0],
    [1.0, 2.0]
])
b = np.array([9.0, 8.0])

# Resolvendo A @ correntes = b sem precisar inverter matriz manualmente
correntes = np.linalg.solve(A, b)
print("Correntes encontradas [i1, i2] (A):", correntes)

# Conferindo a prova real via multiplicação matricial '@' (linha x coluna):
prova_real = A @ correntes
print("Prova real (A @ correntes):", prova_real)

**Análise:** `A @ correntes` retornou `[9., 8.]`, confirmando $3(2) + 1(3) = 9$ e $1(2) + 2(3) = 8$.

---

## 5. Indexação, Fatiamento (*Slicing*) e Máscaras Booleanas

Para extrair pedaços de uma matriz usamos `matriz[linhas, colunas]`:
- `:` sozinho significa **"todas as posições"** daquele eixo.
- `0:2` significa **"do índice 0 até o 1"** (o limite final `2` não entra).
- **Máscara Booleana (`vetor > limite`):** Filtra apenas as medições que atendem a uma condição física!

In [ ]:
# Matriz 3x3 representando sensores (linhas = instantes de tempo, colunas = sensores S0, S1, S2)
M = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

print("Segunda linha inteira (índice 1):", M[1, :])     # Linha 1, todas as colunas
print("Terceira coluna inteira (índice 2):", M[:, 2])   # Todas as linhas, coluna 2
print("Submatriz 2x2 superior esquerda:\n", M[0:2, 0:2])

# Filtragem por máscara booleana: selecionar apenas leituras maiores que 45
leituras_altas = M[M > 45]
print("Leituras acima de 45:", leituras_altas)

## Resumo de Consulta Rápida

| Comando | Função na Engenharia e em IA |
| :--- | :--- |
| `np.array([...])` | Converte listas Python em vetores (1D) ou matrizes (2D). |
| `arr.shape` | Informa as dimensões `(linhas, colunas)` da matriz. |
| `np.linspace(a, b, n)` | Cria malha contínua de `n` pontos entre `a` e `b`. |
| `A @ B` ou `np.dot(A, B)` | Produto escalar / multiplicação matricial linha por coluna. |
| `np.linalg.solve(A, b)` | Resolve o sistema linear $\mathbf{A}\mathbf{x} = \mathbf{b}$. |
| `arr[arr > valor]` | Filtra elementos usando máscara lógica (booleana). |

---

## 6. Exercícios Práticos de Fixação

### Exercício 1: Sinal Linear Calibrado com Ruído Térmico de Sensor
1. Crie um vetor `corrente_ma` com números inteiros de `4` até `20` (inclusive — padrão industrial de instrumentação `4–20 mA`) usando `np.arange()`.
2. Calcule a pressão correspondente em bar sabendo que `pressao_ideal = 0.625 * corrente_ma - 2.5` (onde `4 mA = 0 bar` e `20 mA = 10 bar`).
3. Simule ruído térmico do sensor somando valores aleatórios uniformes entre `-0.1` e `+0.1` bar usando `np.random.default_rng(42)`.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import numpy as np

# 1. De 4 até 20 inclusive exige limite superior 21 no np.arange!
corrente_ma = np.arange(4, 21)

# 2. Broadcasting aplica a equação de reta de calibração no vetor todo
pressao_ideal = 0.625 * corrente_ma - 2.5

# 3. Gerando ruído uniforme entre -0.1 e +0.1 bar com o mesmo tamanho do vetor
rng = np.random.default_rng(42)
ruido = rng.uniform(low=-0.1, high=0.1, size=pressao_ideal.shape)
pressao_medida = pressao_ideal + ruido

print("Corrente (mA):    ", corrente_ma[:5])
print("Pressão ideal:    ", np.round(pressao_ideal[:5], 3))
print("Pressão com ruído:", np.round(pressao_medida[:5], 3))
```

</details>

### Exercício 2: Equilíbrio Estático com Sistema Linear $3 \times 3$
Em uma estrutura mecânica, as forças internas $[F_1, F_2, F_3]$ (em kN) satisfazem o sistema linear:
$$2F_1 + F_2 - F_3 = 8$$
$$-3F_1 - F_2 + 2F_3 = -11$$
$$-2F_1 + F_2 + 2F_3 = -3$$
Monte a matriz `K` ($3 \times 3$) e o vetor `carga` ($3$,), e encontre $[F_1, F_2, F_3]$ usando `np.linalg.solve()`.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import numpy as np

# Matriz de coeficientes das equações de equilíbrio (3x3)
K = np.array([
    [ 2.0,  1.0, -1.0],
    [-3.0, -1.0,  2.0],
    [-2.0,  1.0,  2.0]
])

# Vetor de termos independentes (cargas externas)
carga = np.array([8.0, -11.0, -3.0])

# Resolução direta do sistema K @ forcas = carga
forcas = np.linalg.solve(K, carga)
print("Forças internas [F1, F2, F3] (kN):", forcas)  # Esperado: [2., 3., -1.]
```

</details>

### Exercício 3: O Primeiro Cálculo de um Neurônio Artificial no `NumPy`
Um neurônio recebe 3 medidas normalizadas `x = np.array([0.8, 0.4, 0.2])`, possui pesos `w = np.array([1.5, -2.0, 0.5])` e limiar (*bias*) `theta = 0.3`. Calcule o potencial de ativação $u = (\mathbf{w} \cdot \mathbf{x}) - \theta$ usando `np.dot(w, x)` e imprima se a saída degrau é `1` ($u \ge 0$) ou `0` ($u < 0$).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import numpy as np

# Vetor de entradas e vetor de pesos sinápticos
x = np.array([0.8, 0.4, 0.2])
w = np.array([1.5, -2.0, 0.5])
theta = 0.3  # Limiar de ativação

# Produto escalar (0.8*1.5 + 0.4*(-2.0) + 0.2*0.5) menos o limiar
u = np.dot(w, x) - theta
saida = 1 if u >= 0 else 0

print(f"Potencial u = {u:.2f}  -->  Saída do neurônio = {saida}")
```

</details>

---

## Próximo Passo

Você acabou de ver como o `NumPy` executa a matemática pura das matrizes. Porém, dados reais do mundo industrial chegam em **planilhas com nomes de colunas, textos e números misturados**. Vamos aprender a dominá-las no **[Módulo 02 — Pandas: Exploração, Filtros e Agrupamentos](02_pandas_exploracao_e_agrupamento.ipynb)**!